<a href="https://colab.research.google.com/github/24F2002223/ai_lab/blob/main/Week7_USN1WN24CS107.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### First-Order Logic (FOL) Unification Algorithm

This notebook implements the classic recursive Unification algorithm used in First-Order Logic. Unification is the process of finding a substitution (a set of variable bindings) that makes two logical expressions identical.

In [1]:
class Variable:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return self.name

    def __repr__(self):
        return f"Var({self.name})"

    def __eq__(self, other):
        return isinstance(other, Variable) and self.name == other.name

    def __hash__(self):
        return hash(self.name)


class Constant:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return self.name

    def __repr__(self):
        return f"Const({self.name})"

    def __eq__(self, other):
        return isinstance(other, Constant) and self.name == other.name

    def __hash__(self):
        return hash(self.name)


class Compound:
    """Represents a function or a predicate: e.g., f(x, g(y))"""
    def __init__(self, op, args):
        self.op = op  # operator name (string)
        self.args = list(args)  # list of arguments (Variable, Constant, or Compound)

    def __str__(self):
        return f"{self.op}({', '.join(str(arg) for arg in self.args)})"

    def __repr__(self):
        return f"Compound({self.op}, {repr(self.args)})"

    def __eq__(self, other):
        return isinstance(other, Compound) and self.op == other.op and self.args == other.args

In [2]:
def occur_check(var, expr, theta):
    """Return True if variable var occurs anywhere inside expression expr."""
    if var == expr:
        return True
    elif isinstance(expr, Variable):
        if expr in theta:
            return occur_check(var, theta[expr], theta)
    elif isinstance(expr, Compound):
        return any(occur_check(var, arg, theta) for arg in expr.args)
    return False


def is_variable(expr):
    return isinstance(expr, Variable)


def unify_var(var, val, theta):
    if var in theta:
        return unify(theta[var], val, theta)
    elif is_variable(val) and val in theta:
        return unify(var, theta[val], theta)
    elif occur_check(var, val, theta):
        return None  # Unification fails due to occur-check
    else:
        new_theta = theta.copy()
        new_theta[var] = val
        return new_theta


def unify(x, y, theta=None):
    if theta is None:
        theta = {}

    if theta is None:
        return None
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, Compound) and isinstance(y, Compound):
        if x.op != y.op or len(x.args) != len(y.args):
            return None
        for arg_x, arg_y in zip(x.args, y.args):
            theta = unify(arg_x, arg_y, theta)
            if theta is None:
                return None
        return theta
    elif isinstance(x, list) and isinstance(y, list):
        if len(x) != len(y):
            return None
        for ax, ay in zip(x, y):
            theta = unify(ax, ay, theta)
            if theta is None:
                return None
        return theta
    else:
        return None

In [ ]:
# Demonstration and test cases
x = Variable('x')
y = Variable('y')
a = Constant('a')
b = Constant('b')

# Term 1: f(x, a)
term1 = Compound('f', [x, a])
# Term 2: f(b, y)
term2 = Compound('f', [b, y])

print(f"Unifying: {term1} and {term2}")
sub = unify(term1, term2)
if sub is not None:
    formatted_sub = {str(k): str(v) for k, v in sub.items()}
    print("Success! Substitution:", formatted_sub)
else:
    print("Failed to unify.")

Unifying: f(x, a) and f(b, y)
Success! Substitution: {'x': 'b', 'y': 'a'}
